In [1]:
conda install pytorch torchvision torchaudio -c pytorch

Channels:
 - pytorch
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image
import csv

In [ ]:

# ------------------------------
# 1. Data Preparation
# ------------------------------
transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('data/train', transform_train)
val_dataset = datasets.ImageFolder('data/valid', transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ------------------------------
# 2. Model
# ------------------------------
model = models.resnet50(pretrained=True)
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # 2 classes: cat, dog

if not torch.backends.mps.is_available():
    print("No GPU (MPS) available. Exiting.")
    exit()
else:
    device = torch.device('mps')
    print("Using MPS (Metal Performance Shaders) for GPU acceleration.")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ------------------------------
# 3. Training
# ------------------------------
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_acc = 100 * correct / total

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        val_acc = 100 * val_correct / val_total

        print(f'Epoch {epoch+1}/{epochs} Loss: {running_loss/len(train_loader):.4f} '
              f'Train Acc: {train_acc:.2f}% Val Acc: {val_acc:.2f}%')

train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10)

# ------------------------------
# 4. Fine-tuning (optional)
# ------------------------------
# Unfreeze some layers and continue training with lower lr
for param in model.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=1e-5)
train_model(model, criterion, optimizer, train_loader, val_loader, epochs=5)

# Save model
torch.save(model.state_dict(), 'cat_dog_classifier.pth')


/Users/nawfalabdulmalick/miniconda3/envs/daenv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nawfalabdulmalick/miniconda3/envs/daenv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using MPS (Metal Performance Shaders) for GPU acceleration.


In [ ]:


# Define device (use MPS if available)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Recreate the same model structure
model = models.resnet50(pretrained=False)  # we'll load saved weights
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)          # 2 classes: cat, dog
model.load_state_dict(torch.load('cat_dog_classifier.pth', map_location=device))
model = model.to(device)
model.eval()  # set to evaluation mode

/Users/nawfalabdulmalick/miniconda3/envs/myenv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/var/folders/0q/q6vx57391lg0gkgbn1m9flrr0000gn/T/ipykernel_21966/1034421240.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explic

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
# Preprocessing pipeline (same as validation) 
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict_image(image_path):
    """Return predicted class and confidence for a single image."""
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)  # add batch dimension

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    class_names = ['cat', 'dog']   # order from ImageFolder: classes[0]='cat', [1]='dog'
    return class_names[predicted.item()], confidence.item()

# Folder containing test images
test_folder = 'data/test1'

# Get all image files (common extensions)
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith(image_extensions)]

if not image_files:
    print("No image files found in the folder.")
else:
    print(f"Found {len(image_files)} images. Processing...\n")
    for img_file in image_files:
        img_path = os.path.join(test_folder, img_file)
        label, conf = predict_image(img_path)
        print(f"{img_file}: {label} (confidence: {conf:.2%})")

Found 12500 images. Processing...

9733.jpg: cat (confidence: 100.00%)
63.jpg: cat (confidence: 100.00%)
6400.jpg: dog (confidence: 100.00%)
823.jpg: dog (confidence: 100.00%)
4217.jpg: cat (confidence: 99.81%)
3578.jpg: cat (confidence: 100.00%)
10321.jpg: dog (confidence: 100.00%)
2666.jpg: cat (confidence: 99.98%)
5109.jpg: cat (confidence: 100.00%)
11981.jpg: cat (confidence: 99.77%)


In [ ]:

results = []
for img_file in image_files:
    img_path = os.path.join(test_folder, img_file)
    label, conf = predict_image(img_path)
    results.append([img_file, label, conf])

with open('predictions.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filename', 'predicted_class', 'confidence'])
    writer.writerows(results)

print("Predictions saved to predictions.csv")

Predictions saved to predictions.csv
